In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import json
import os
from datetime import datetime
import joblib  # More efficient than pickle for sklearn models
import shap
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import LeaveOneOut, cross_val_predict, train_test_split, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
import plotly.graph_objects as go
import plotly.express as px
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
import seaborn as sns
from tqdm import tqdm
import warnings
from model_pipeline import train_and_save_model

warnings.filterwarnings('ignore', category=UserWarning)

/usr/local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

def save_model_package(results, output_dir, model_name="pathogenicity_model"):
    """
    Save a complete model package that can be easily loaded and used for predictions
    
    Parameters:
    -----------
    results : dict
        Results dictionary from train_pathogenicity_model
    output_dir : str
        Directory to save the model package
    model_name : str
        Name for the model package
        
    Returns:
    --------
    str: Path to the saved model package
    """
    # Create model package directory
    package_dir = os.path.join(output_dir, f"{model_name}_package")
    os.makedirs(package_dir, exist_ok=True)
    
    print(f"Saving model package to {package_dir}...")
    
    # 1. Save models using joblib (more efficient for sklearn models)
    models_dir = os.path.join(package_dir, "models")
    os.makedirs(models_dir, exist_ok=True)
    
    for model_name, model in results['models'].items():
        model_path = os.path.join(models_dir, f"{model_name}.joblib")
        joblib.dump(model, model_path)
        print(f"  Saved {model_name} model")
    
    # 2. Save explainers
    explainers_dir = os.path.join(package_dir, "explainers")
    os.makedirs(explainers_dir, exist_ok=True)
    
    for explainer_name, explainer in results['explainers'].items():
        explainer_path = os.path.join(explainers_dir, f"{explainer_name}_explainer.pkl")
        with open(explainer_path, 'wb') as f:
            pickle.dump(explainer, f)
        print(f"  Saved {explainer_name} explainer")
    
    # 3. Save metadata as JSON (human-readable)
    metadata = {
        'creation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'feature_names': results['feature_names'],
        'class_names': results['class_names'],
        'num_classes': results['num_classes'],
        'binary_threshold': results.get('binary_threshold'),
        'model_types': list(results['models'].keys()),
        'num_features': len(results['feature_names']),
        'cv_results': {
            model_name: {
                'accuracy': float(metrics['accuracy']),
                'f1_macro': float(metrics['f1_macro']),
                'f1_weighted': float(metrics['f1_weighted'])
            }
            for model_name, metrics in results['cv_results'].items()
        }
    }
    
    metadata_path = os.path.join(package_dir, "metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"  Saved metadata")
    
    # 4. Create a requirements.txt file
    requirements = [
        "numpy",
        "pandas",
        "scikit-learn",
        "xgboost",
        "catboost",
        "shap",
        "joblib",
        "matplotlib",
        "seaborn"
    ]
    
    requirements_path = os.path.join(package_dir, "requirements.txt")
    with open(requirements_path, 'w') as f:
        f.write('\n'.join(requirements))
    print(f"  Saved requirements.txt")
    
    # 5. Create a README
    readme_content = f"""# Pathogenicity Prediction Model Package

Created: {metadata['creation_date']}

## Model Information
- Number of features: {metadata['num_features']}
- Number of classes: {metadata['num_classes']}
- Class names: {', '.join(metadata['class_names'])}
- Available models: {', '.join(metadata['model_types'])}

## Model Performance
"""
    
    for model_name, metrics in metadata['cv_results'].items():
        readme_content += f"\n### {model_name.upper()}\n"
        readme_content += f"- Accuracy: {metrics['accuracy']:.4f}\n"
        readme_content += f"- F1 Macro: {metrics['f1_macro']:.4f}\n"
        readme_content += f"- F1 Weighted: {metrics['f1_weighted']:.4f}\n"
    
    readme_content += """
## Usage

```python
from predict_pathogenicity import PathogenicityPredictor

# Load the model
predictor = PathogenicityPredictor('path/to/model_package')

# Predict on new data
results = predictor.predict('path/to/new_genome.csv', model_type='ensemble')

# Get detailed explanation
explanation = predictor.explain_prediction('path/to/new_genome.csv', model_type='xgb')
```

## Files
- `models/`: Trained model files (.joblib)
- `explainers/`: SHAP explainers (.pkl)
- `metadata.json`: Model metadata and performance metrics
- `requirements.txt`: Python package requirements
- `predict_pathogenicity.py`: Standalone prediction script
"""
    
    readme_path = os.path.join(package_dir, "README.md")
    with open(readme_path, 'w') as f:
        f.write(readme_content)
    print(f"  Saved README.md")
    
    print(f"\nModel package saved successfully to: {package_dir}")
    return package_dir


def create_standalone_predictor(package_dir):
    """
    Create a standalone prediction script in the model package
    
    Parameters:
    -----------
    package_dir : str
        Path to the model package directory
    """
    predictor_code = '''\
\"\"\"
Standalone Pathogenicity Predictor

This script provides a simple interface for loading the trained model
and making predictions on new genome data.
\"\"\"

import numpy as np
import pandas as pd
import joblib
import pickle
import json
import os
import shap
from typing import Dict, Optional, Union
import warnings
warnings.filterwarnings('ignore')



class PathogenicityPredictor:
    """
    A class to load trained models and make predictions on new genome data
    """
    
    def __init__(self, model_package_path: str):
        """
        Initialize the predictor by loading the model package
        
        Parameters:
        -----------
        model_package_path : str
            Path to the model package directory
        """
        self.package_path = model_package_path
        self.models = {}
        self.explainers = {}
        self.metadata = None
        
        # Load metadata
        metadata_path = os.path.join(model_package_path, "metadata.json")
        with open(metadata_path, 'r') as f:
            self.metadata = json.load(f)
        
        print(f"Loading model package from {model_package_path}")
        print(f"Created: {self.metadata['creation_date']}")
        print(f"Features: {self.metadata['num_features']}")
        print(f"Classes: {self.metadata['num_classes']}")
        
        # Load all models
        models_dir = os.path.join(model_package_path, "models")
        for model_file in os.listdir(models_dir):
            if model_file.endswith('.joblib'):
                model_name = model_file.replace('.joblib', '')
                model_path = os.path.join(models_dir, model_file)
                self.models[model_name] = joblib.load(model_path)
                print(f"  Loaded {model_name} model")
        
        # Load explainers
        explainers_dir = os.path.join(model_package_path, "explainers")
        if os.path.exists(explainers_dir):
            for explainer_file in os.listdir(explainers_dir):
                if explainer_file.endswith('.pkl'):
                    explainer_name = explainer_file.replace('_explainer.pkl', '')
                    explainer_path = os.path.join(explainers_dir, explainer_file)
                    with open(explainer_path, 'rb') as f:
                        self.explainers[explainer_name] = pickle.load(f)
                    print(f"  Loaded {explainer_name} explainer")
    
    def _prepare_features(self, genome_data: Union[str, pd.DataFrame]) -> pd.DataFrame:
        """
        Prepare feature matrix from genome data
        
        Parameters:
        -----------
        genome_data : str or pd.DataFrame
            Path to CSV file or DataFrame with genome data
            
        Returns:
        --------
        pd.DataFrame: Prepared feature matrix
        """
        # Load data if it's a path
        if isinstance(genome_data, str):
            df = pd.read_csv(genome_data, index_col=0)
        else:
            df = genome_data.copy()
        
        # Ensure all required features are present
        required_features = self.metadata['feature_names']
        missing_features = [f for f in required_features if f not in df.columns]
        
        if missing_features:
            print(f"Warning: {len(missing_features)} features missing from input data")
            print(f"Adding missing features with value 0")
            for feature in missing_features:
                df[feature] = 0
        
        # Select only the features used by the model, in the correct order
        X = df[required_features]
        
        return X
    
    def predict(self, 
                genome_data: Union[str, pd.DataFrame],
                model_type: str = 'ensemble',
                return_probabilities: bool = True) -> Dict:
        """
        Make prediction on new genome data
        
        Parameters:
        -----------
        genome_data : str or pd.DataFrame
            Path to CSV file or DataFrame with genome data
        model_type : str
            Which model to use ('xgb', 'catboost', 'rf', 'ensemble')
        return_probabilities : bool
            Whether to return probability distributions
            
        Returns:
        --------
        dict: Prediction results
        """
        # Prepare features
        X = self._prepare_features(genome_data)
        
        # Select model
        if model_type not in self.models:
            raise ValueError(f"Model '{model_type}' not found. Available: {list(self.models.keys())}")
        
        model = self.models[model_type]
        
        # Make predictions
        predictions = model.predict(X)
        
        results = {
            'predictions': predictions.tolist(),
            'predicted_classes': [self.metadata['class_names'][int(p)] for p in predictions],
            'model_used': model_type
        }
        
        # Add probabilities if requested
        if return_probabilities:
            probabilities = model.predict_proba(X)
            results['probabilities'] = probabilities.tolist()
            
            # Add probability dict for each sample
            results['probability_distributions'] = [
                {class_name: float(prob) 
                 for class_name, prob in zip(self.metadata['class_names'], probs)}
                for probs in probabilities
            ]
        
        return results
    
    def explain_prediction(self,
                          genome_data: Union[str, pd.DataFrame],
                          model_type: str = 'xgb',
                          sample_idx: int = 0,
                          top_n_features: int = 20) -> Dict:
        """
        Get SHAP explanation for a prediction
        
        Parameters:
        -----------
        genome_data : str or pd.DataFrame
            Path to CSV file or DataFrame with genome data
        model_type : str
            Which model to use for explanation ('xgb', 'catboost', 'rf')
        sample_idx : int
            Index of sample to explain
        top_n_features : int
            Number of top features to include in explanation
            
        Returns:
        --------
        dict: Explanation with SHAP values and top features
        """
        # Prepare features
        X = self._prepare_features(genome_data)
        
        # Get explainer
        if model_type not in self.explainers:
            raise ValueError(f"Explainer for '{model_type}' not found. Available: {list(self.explainers.keys())}")
        
        if model_type not in self.models:
            raise ValueError(f"Model '{model_type}' not found")
        
        explainer = self.explainers[model_type]
        model = self.models[model_type]
        
        # Make prediction
        prediction = model.predict(X[sample_idx:sample_idx+1])[0]
        probabilities = model.predict_proba(X[sample_idx:sample_idx+1])[0]
        
        # Calculate SHAP values
        shap_values = explainer.shap_values(X[sample_idx:sample_idx+1])
        
        # Handle different SHAP value formats
        if isinstance(shap_values, list):
            # Multi-class: use values for predicted class
            relevant_shap = shap_values[prediction][0]
        else:
            # Binary: use as is
            relevant_shap = shap_values[0]
        
        # Get top features
        feature_importance = list(zip(
            self.metadata['feature_names'],
            X.iloc[sample_idx].values,
            relevant_shap
        ))
        
        # Sort by absolute SHAP value
        feature_importance.sort(key=lambda x: abs(x[2]), reverse=True)
        top_features = feature_importance[:top_n_features]
        
        explanation = {
            'sample_index': sample_idx,
            'prediction': int(prediction),
            'predicted_class': self.metadata['class_names'][prediction],
            'probabilities': {
                class_name: float(prob)
                for class_name, prob in zip(self.metadata['class_names'], probabilities)
            },
            'top_features': [
                {
                    'feature': feat,
                    'value': float(val),
                    'shap_value': float(shap_val),
                    'impact': 'positive' if shap_val > 0 else 'negative'
                }
                for feat, val, shap_val in top_features
            ]
        }
        
        return explanation
    
    def batch_predict(self,
                     genome_data: Union[str, pd.DataFrame],
                     model_type: str = 'ensemble',
                     output_file: Optional[str] = None) -> pd.DataFrame:
        """
        Make predictions on multiple genomes and return as DataFrame
        
        Parameters:
        -----------
        genome_data : str or pd.DataFrame
            Path to CSV file or DataFrame with genome data
        model_type : str
            Which model to use
        output_file : str, optional
            Path to save results CSV
            
        Returns:
        --------
        pd.DataFrame: Results with predictions and probabilities
        """
        # Get predictions
        results = self.predict(genome_data, model_type, return_probabilities=True)
        
        # Create DataFrame
        df_results = pd.DataFrame({
            'prediction': results['predictions'],
            'predicted_class': results['predicted_classes']
        })
        
        # Add probability columns
        for i, class_name in enumerate(self.metadata['class_names']):
            df_results[f'prob_{class_name}'] = [probs[i] for probs in results['probabilities']]
        
        # Save if requested
        if output_file:
            df_results.to_csv(output_file, index=False)
            print(f"Results saved to {output_file}")
        
        return df_results
    
    def get_model_info(self) -> Dict:
        """
        Get information about the loaded model
        
        Returns:
        --------
        dict: Model metadata and performance metrics
        """
        return self.metadata


# Command-line interface
if __name__ == "__main__":
    import argparse
    
    parser = argparse.ArgumentParser(description='Predict pathogenicity for new genomes')
    parser.add_argument('--model-package', required=True, help='Path to model package directory')
    parser.add_argument('--input', required=True, help='Path to input CSV file with genome data')
    parser.add_argument('--output', help='Path to save output CSV file')
    parser.add_argument('--model-type', default='ensemble', 
                       choices=['xgb', 'catboost', 'rf', 'ensemble'],
                       help='Which model to use for prediction')
    parser.add_argument('--explain', action='store_true', 
                       help='Generate SHAP explanations for first sample')
    
    args = parser.parse_args()
    
    # Load predictor
    predictor = PathogenicityPredictor(args.model_package)
    
    # Make predictions
    print(f"\\nMaking predictions using {args.model_type} model...")
    results = predictor.batch_predict(args.input, args.model_type, args.output)
    
    print(f"\\nPredictions for {len(results)} samples:")
    print(results.head(10))
    
    # Generate explanation if requested
    if args.explain:
        print("\\nGenerating explanation for first sample...")
        explanation = predictor.explain_prediction(args.input, args.model_type, sample_idx=0)
        
        print(f"\\nPrediction: {explanation['predicted_class']}")
        print(f"Probabilities: {explanation['probabilities']}")
        print(f"\\nTop 10 features contributing to prediction:")
        for i, feat_info in enumerate(explanation['top_features'][:10], 1):
            print(f"{i}. {feat_info['feature'][:50]}")
            print(f"   Value: {feat_info['value']:.4f}, "
                  f"SHAP: {feat_info['shap_value']:.4f} ({feat_info['impact']})")
'''
    
    # Save the predictor script
    predictor_path = os.path.join(package_dir, "predict_pathogenicity.py")
    with open(predictor_path, 'w') as f:
        f.write(predictor_code)
    
    print(f"Created standalone prediction script: {predictor_path}")
    print("\nYou can now use this script to make predictions:")
    print(f"  python {predictor_path} --model-package {package_dir} --input new_genome.csv --output predictions.csv")


# Update your training function to automatically create the model package
def train_and_save_model(data_path, output_dir, **train_kwargs):
    """
    Train model and save as a complete package
    
    Parameters:
    -----------
    data_path : str
        Path to training data
    output_dir : str
        Output directory
    **train_kwargs : dict
        Additional arguments for train_pathogenicity_model
        
    Returns:
    --------
    tuple: (results dict, model package path)
    """
    # Import your training function
    from your_module import train_pathogenicity_model
    
    # Train the model
    print("Training model...")
    results = train_pathogenicity_model(
        data_path=data_path,
        output_dir=output_dir,
        **train_kwargs
    )
    
    # Save as package
    print("\nCreating model package...")
    package_path = save_model_package(results, output_dir)
    
    # Create standalone predictor
    create_standalone_predictor(package_path)
    
    return results, package_path

In [3]:
def generate_interactive_waterfall_plot(
    sample, 
    true_class, 
    sample_idx, 
    explainer, 
    shap_values, 
    feature_names, 
    class_names, 
    output_dir, 
    num_classes
):
    """
    Generate interactive waterfall plot for a specific sample using Plotly
    
    Parameters:
    -----------
    sample : pandas Series
        Single sample to explain
    true_class : int
        Actual class of the sample
    sample_idx : int
        Index of the sample in the dataset
    explainer : shap.TreeExplainer
        SHAP explainer object
    shap_values : list or ndarray
        Precomputed SHAP values
    feature_names : list
        List of feature names
    class_names : list
        List of class names
    output_dir : str
        Directory to save plots
    num_classes : int
        Number of classes
    """
    import plotly.graph_objects as go
    
    # Create a directory for waterfall plots if it doesn't exist
    waterfall_dir = os.path.join(output_dir, "interactive_waterfall_plots")
    os.makedirs(waterfall_dir, exist_ok=True)
    
    if num_classes == 2:
        # For binary classification, plot the positive class
        
        # Handle different shap_values formats
        if isinstance(shap_values, list):
            # For XGBoost binary classification, positive class is index 1
            values = shap_values[1][sample_idx]
            base_value = explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value
        else:
            # For models that return a single array
            values = shap_values[sample_idx]
            base_value = explainer.expected_value
        
        # Sort features by absolute SHAP value
        indices = np.argsort(np.abs(values))
        
        # Get the top features (limit to 20 for readability)
        top_n = min(20, len(indices))
        indices = indices[-top_n:]
        
        # Extract the relevant features and values
        features = [feature_names[i] for i in indices]
        feature_values = [sample.values[i] for i in indices]
        shap_values_sorted = values[indices]
        
        # Create data for visualization
        cumulative_values = np.cumsum(shap_values_sorted)
        waterfall_data = []
        
        # Add base value
        waterfall_data.append({
            'feature': 'Base value',
            'shap_value': 0,
            'cumulative': base_value,
            'feature_value': None,
            'color': 'gray'
        })
        
        # Add each feature's contribution
        for i, (feature, value, shap_value) in enumerate(zip(features, feature_values, shap_values_sorted)):
            waterfall_data.append({
                'feature': feature,
                'shap_value': shap_value,
                'cumulative': base_value + cumulative_values[i],
                'feature_value': value,
                'color': 'red' if shap_value < 0 else 'blue'
            })
        
        # Add final prediction
        waterfall_data.append({
            'feature': 'Final prediction',
            'shap_value': 0,
            'cumulative': base_value + cumulative_values[-1],
            'feature_value': None,
            'color': 'gray'
        })
        
        # Create DataFrame for plotting
        import pandas as pd
        df = pd.DataFrame(waterfall_data)
        
        # Create the waterfall plot
        fig = go.Figure(go.Waterfall(
            name="SHAP",
            orientation="h",
            measure=["absolute"] + ["relative"]*len(features) + ["total"],
            x=df['shap_value'],
            y=df['feature'],
            connector={"line": {"color": "rgb(63, 63, 63)"}},
            text=[f"{x:.3f}" for x in df['shap_value']],
            textposition="outside",
            decreasing={"marker": {"color": "red"}},
            increasing={"marker": {"color": "blue"}},
        ))
        
        fig.update_layout(
            title=f"Sample {sample_idx} - True Class: {class_names[true_class]}",
            xaxis_title="SHAP Value",
            yaxis_title="Feature",
            height=600,
            width=900,
            showlegend=False,
            annotations=[
                dict(
                    x=base_value,
                    y="Base value",
                    text=f"Base value: {base_value:.3f}",
                    showarrow=False,
                    font=dict(size=12),
                    xanchor="left"
                ),
                dict(
                    x=base_value + cumulative_values[-1],
                    y="Final prediction",
                    text=f"Final: {(base_value + cumulative_values[-1]):.3f}",
                    showarrow=False,
                    font=dict(size=12),
                    xanchor="left"
                )
            ]
        )
        
        # Add hover information for feature values
        hover_info = []
        for i, row in df.iterrows():
            if row['feature_value'] is not None:
                hover_info.append(f"Feature value: {row['feature_value']}")
            else:
                hover_info.append("")
        
        fig.update_traces(hovertext=hover_info)
        
        # Save the figure as HTML
        fig.write_html(f"{waterfall_dir}/interactive_waterfall_sample_{sample_idx}.html")
        
    else:
        # For multi-class, create a waterfall plot for each class
        for class_idx in range(num_classes):
            # Extract the SHAP values for this class and sample
            if isinstance(shap_values, list):
                values = shap_values[class_idx][sample_idx]
                base_value = explainer.expected_value[class_idx]
            else:
                # Handle case where shap_values is a 3D array (samples x features x classes)
                if shap_values.ndim == 3:
                    values = shap_values[sample_idx, :, class_idx]
                    base_value = explainer.expected_value[class_idx] if hasattr(explainer.expected_value, '__len__') else explainer.expected_value
                else:
                    # Fallback for 2D array
                    values = shap_values[sample_idx]
                    base_value = explainer.expected_value
            
            # Sort features by absolute SHAP value
            indices = np.argsort(np.abs(values))
            
            # Get the top features (limit to 20 for readability)
            top_n = min(20, len(indices))
            indices = indices[-top_n:]
            
            # Extract the relevant features and values
            features = [feature_names[i] for i in indices]
            feature_values = [sample.values[i] for i in indices]
            shap_values_sorted = values[indices]
            
            # Create data for visualization
            cumulative_values = np.cumsum(shap_values_sorted)
            waterfall_data = []
            
            # Add base value
            waterfall_data.append({
                'feature': 'Base value',
                'shap_value': 0,
                'cumulative': base_value,
                'feature_value': None,
                'color': 'gray'
            })
            
            # Add each feature's contribution
            for i, (feature, value, shap_value) in enumerate(zip(features, feature_values, shap_values_sorted)):
                waterfall_data.append({
                    'feature': feature,
                    'shap_value': shap_value,
                    'cumulative': base_value + cumulative_values[i],
                    'feature_value': value,
                    'color': 'red' if shap_value < 0 else 'blue'
                })
            
            # Add final prediction
            waterfall_data.append({
                'feature': 'Final prediction',
                'shap_value': 0,
                'cumulative': base_value + cumulative_values[-1],
                'feature_value': None,
                'color': 'gray'
            })
            
            # Create DataFrame for plotting
            import pandas as pd
            df = pd.DataFrame(waterfall_data)
            
            # Create the waterfall plot
            fig = go.Figure(go.Waterfall(
                name="SHAP",
                orientation="h",
                measure=["absolute"] + ["relative"]*len(features) + ["total"],
                x=df['shap_value'],
                y=df['feature'],
                connector={"line": {"color": "rgb(63, 63, 63)"}},
                text=[f"{x:.3f}" for x in df['shap_value']],
                textposition="outside",
                decreasing={"marker": {"color": "red"}},
                increasing={"marker": {"color": "blue"}},
            ))
            
            fig.update_layout(
                title=f"Sample {sample_idx} - True Class: {class_names[true_class]} - Impact on {class_names[class_idx]}",
                xaxis_title="SHAP Value",
                yaxis_title="Feature",
                height=600,
                width=900,
                showlegend=False,
                annotations=[
                    dict(
                        x=base_value,
                        y="Base value",
                        text=f"Base value: {base_value:.3f}",
                        showarrow=False,
                        font=dict(size=12),
                        xanchor="left"
                    ),
                    dict(
                        x=base_value + cumulative_values[-1],
                        y="Final prediction",
                        text=f"Final: {(base_value + cumulative_values[-1]):.3f}",
                        showarrow=False,
                        font=dict(size=12),
                        xanchor="left"
                    )
                ]
            )
            
            # Add hover information for feature values
            hover_info = []
            for i, row in df.iterrows():
                if row['feature_value'] is not None:
                    hover_info.append(f"Feature value: {row['feature_value']}")
                else:
                    hover_info.append("")
            
            fig.update_traces(hovertext=hover_info)
            
            # Save the figure as HTML
            fig.write_html(f"{waterfall_dir}/interactive_waterfall_sample_{sample_idx}_class_{class_idx}.html")

In [4]:
def cluster_features(X, feature_names, n_clusters=10, method='cosine'):
    """
    Cluster features based on their patterns across samples
    
    Parameters:
    -----------
    X : pandas DataFrame
        Feature matrix
    feature_names : list
        List of feature names
    n_clusters : int
        Number of clusters to create
    method : str
        Similarity method ('cosine' or 'correlation')
        
    Returns:
    --------
    dict: Dictionary mapping features to clusters, and cluster info
    """
    # Calculate feature similarity matrix
    if method == 'cosine':
        similarity_matrix = cosine_similarity(X.T)
    else:
        # Correlation-based similarity
        similarity_matrix = np.corrcoef(X.T)
    
    # Perform hierarchical clustering
    clustering = AgglomerativeClustering(
        n_clusters=n_clusters,
        affinity='precomputed',
        linkage='average'
    )
    
    # Convert similarity to distance (1 - similarity)
    distance_matrix = 1 - similarity_matrix
    
    # Fit clustering
    cluster_labels = clustering.fit_predict(distance_matrix)
    
    # Create mapping from feature to cluster
    feature_cluster_map = {feature: cluster for feature, cluster in zip(feature_names, cluster_labels)}
    
    # Create cluster info
    cluster_info = {}
    for cluster_id in range(n_clusters):
        cluster_features = [feature for feature, cluster in feature_cluster_map.items() if cluster == cluster_id]
        cluster_info[cluster_id] = {
            'features': cluster_features,
            'size': len(cluster_features)
        }
    
    # Create visualization of clusters
    fig = px.imshow(
        similarity_matrix,
        labels=dict(x="Features", y="Features", color="Similarity"),
        x=feature_names,
        y=feature_names,
        color_continuous_scale="Viridis"
    )
    
    # Add clustering information
    cluster_boundaries = []
    sorted_indices = np.argsort(cluster_labels)
    sorted_features = [feature_names[i] for i in sorted_indices]
    
    # Reorder the matrix based on clusters
    reordered_matrix = similarity_matrix[sorted_indices, :][:, sorted_indices]
    
    # Create a new figure with reordered matrix
    fig_reordered = px.imshow(
        reordered_matrix,
        labels=dict(x="Features", y="Features", color="Similarity"),
        x=sorted_features,
        y=sorted_features,
        color_continuous_scale="Viridis"
    )
    
    # Add cluster boundary lines
    current_pos = 0
    for cluster_id in range(n_clusters):
        cluster_size = len([f for f in feature_names if feature_cluster_map[f] == cluster_id])
        if cluster_size > 0:
            current_pos += cluster_size
            # Add vertical line
            fig_reordered.add_shape(
                type="line",
                x0=current_pos - 0.5,
                x1=current_pos - 0.5,
                y0=-0.5,
                y1=len(feature_names) - 0.5,
                line=dict(color="black", width=2)
            )
            # Add horizontal line
            fig_reordered.add_shape(
                type="line",
                x0=-0.5,
                x1=len(feature_names) - 0.5,
                y0=current_pos - 0.5,
                y1=current_pos - 0.5,
                line=dict(color="black", width=2)
            )
    
    fig_reordered.update_layout(
        title="Feature Similarity Matrix (Sorted by Cluster)",
        height=800,
        width=900
    )
    
    return {
        'feature_cluster_map': feature_cluster_map,
        'cluster_info': cluster_info,
        'similarity_matrix': similarity_matrix,
        'visualization': fig_reordered
    }

In [5]:
def analyze_gene_clusters(model_dir, n_clusters=10, method='cosine'):
    """
    Analyze gene clusters based on the trained model data
    
    Parameters:
    -----------
    model_dir : str
        Directory containing model data
    n_clusters : int
        Number of clusters to create
    method : str
        Similarity method ('cosine' or 'correlation')
        
    Returns:
    --------
    dict: Clustering results
    """
    # Create output directory for clusters
    clusters_dir = os.path.join(model_dir, "gene_clusters")
    os.makedirs(clusters_dir, exist_ok=True)
    
    # Load the dataset
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    if not os.path.exists(dataset_path):
        print(f"Warning: Dataset not found at {dataset_path}. Please provide the correct path.")
        return None
    
    dataset = pd.read_csv(dataset_path, index_col=0)
    
    # Load feature names
    with open(os.path.join(model_dir, "feature_names.pkl"), 'rb') as f:
        feature_names = pickle.load(f)
    
    # Load class info
    with open(os.path.join(model_dir, "class_info.pkl"), 'rb') as f:
        class_info = pickle.load(f)
    
    # Extract features and target
    X = dataset[feature_names]
    target_column = [col for col in dataset.columns if col not in feature_names][0]
    y = dataset[target_column].values
    
    # Perform clustering
    print(f"Clustering {len(feature_names)} genes into {n_clusters} clusters...")
    cluster_results = cluster_features(X, feature_names, n_clusters, method)
    
    # Save cluster visualization
    cluster_results['visualization'].write_html(os.path.join(clusters_dir, "gene_clusters.html"))
    
    # Analyze importance by cluster
    # Load models and explainers
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    with open(os.path.join(model_dir, "shap_explainers.pkl"), 'rb') as f:
        explainers = pickle.load(f)
    
    # Use XGBoost model for SHAP analysis
    model = models.get('xgb')
    explainer = explainers.get('xgb')
    
    if model and explainer:
        # Calculate SHAP values for all samples
        shap_values = explainer.shap_values(X)
        
        # Analyze importance by cluster
        cluster_importance = {}
        
        if isinstance(shap_values, list):  # Multi-class case
            for class_idx in range(class_info['num_classes']):
                cluster_importance[class_idx] = {}
                
                for cluster_id, cluster_data in cluster_results['cluster_info'].items():
                    # Get feature indices for this cluster
                    feature_indices = [feature_names.index(feature) for feature in cluster_data['features']]
                    
                    # Calculate mean absolute SHAP value for features in this cluster
                    cluster_shap = np.abs(shap_values[class_idx][:, feature_indices]).mean()
                    
                    cluster_importance[class_idx][cluster_id] = {
                        'mean_abs_shap': cluster_shap,
                        'n_features': len(feature_indices)
                    }
        else:  # Binary classification case
            for cluster_id, cluster_data in cluster_results['cluster_info'].items():
                # Get feature indices for this cluster
                feature_indices = [feature_names.index(feature) for feature in cluster_data['features']]
                
                # Calculate mean absolute SHAP value for features in this cluster
                cluster_shap = np.abs(shap_values[:, feature_indices]).mean()
                
                cluster_importance[0][cluster_id] = {
                    'mean_abs_shap': cluster_shap,
                    'n_features': len(feature_indices)
                }
        
        # Save cluster importance information
        with open(os.path.join(clusters_dir, "cluster_importance.pkl"), 'wb') as f:
            pickle.dump(cluster_importance, f)
        
        # Create visualization of cluster importance
        cluster_imp_viz = {}
        
        if isinstance(shap_values, list):  # Multi-class case
            for class_idx in range(class_info['num_classes']):
                # Prepare data for visualization
                data = []
                for cluster_id, imp_data in cluster_importance[class_idx].items():
                    data.append({
                        'cluster_id': f"Cluster {cluster_id}",
                        'mean_abs_shap': imp_data['mean_abs_shap'],
                        'n_features': imp_data['n_features']
                    })
                
                df = pd.DataFrame(data)
                
                # Create bar plot
                fig = px.bar(
                    df,
                    x='cluster_id',
                    y='mean_abs_shap',
                    color='n_features',
                    labels={'mean_abs_shap': 'Mean Absolute SHAP Value', 'cluster_id': 'Gene Cluster'},
                    title=f"Cluster Importance for Class {class_info['class_names'][class_idx]}"
                )
                
                fig.update_layout(height=600, width=900)
                fig.write_html(os.path.join(clusters_dir, f"cluster_importance_class_{class_idx}.html"))
                
                cluster_imp_viz[class_idx] = fig
        else:  # Binary classification case
            # Prepare data for visualization
            data = []
            for cluster_id, imp_data in cluster_importance[0].items():
                data.append({
                    'cluster_id': f"Cluster {cluster_id}",
                    'mean_abs_shap': imp_data['mean_abs_shap'],
                    'n_features': imp_data['n_features']
                })
            
            df = pd.DataFrame(data)
            
            # Create bar plot
            fig = px.bar(
                df,
                x='cluster_id',
                y='mean_abs_shap',
                color='n_features',
                labels={'mean_abs_shap': 'Mean Absolute SHAP Value', 'cluster_id': 'Gene Cluster'},
                title="Cluster Importance"
            )
            
            fig.update_layout(height=600, width=900)
            fig.write_html(os.path.join(clusters_dir, "cluster_importance.html"))
            
            cluster_imp_viz[0] = fig
        
        cluster_results['cluster_importance'] = cluster_importance
        cluster_results['importance_visualizations'] = cluster_imp_viz
    
    # Save complete clustering results
    with open(os.path.join(clusters_dir, "clustering_results.pkl"), 'wb') as f:
        pickle.dump(cluster_results, f)
    
    # Create detailed report for each cluster
    cluster_report = []
    
    for cluster_id, cluster_data in cluster_results['cluster_info'].items():
        report = {}
        report['cluster_id'] = cluster_id
        report['n_features'] = len(cluster_data['features'])
        report['features'] = cluster_data['features']
        
        # Add importance information if available
        if 'cluster_importance' in cluster_results:
            if isinstance(shap_values, list):  # Multi-class case
                importance_by_class = {}
                for class_idx in range(class_info['num_classes']):
                    importance_by_class[class_idx] = cluster_results['cluster_importance'][class_idx][cluster_id]['mean_abs_shap']
                report['importance_by_class'] = importance_by_class
            else:  # Binary classification case
                report['importance'] = cluster_results['cluster_importance'][0][cluster_id]['mean_abs_shap']
        
        cluster_report.append(report)
    
    # Create a DataFrame for the report
    cluster_df = pd.DataFrame(cluster_report)
    
    # Save as CSV
    cluster_df.to_csv(os.path.join(clusters_dir, "cluster_summary.csv"), index=False)
    
    print(f"Gene clustering completed. Results saved to {clusters_dir}")
    
    return cluster_results

In [6]:
def check_gpu_status():
    """
    Thoroughly check GPU availability and configuration for different ML libraries
    
    Returns:
    --------
    dict: Status of GPU for each major library
    """
    status = {
        "xgboost_gpu": False,
        "catboost_gpu": False,
        "system_cuda": False,
        "gpu_device_name": None,
        "gpu_memory_gb": None
    }
    
    # Check system CUDA
    try:
        import subprocess
        result = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE)
        if result.returncode == 0:
            status["system_cuda"] = True
            # Try to get GPU name and memory
            try:
                import torch
                if torch.cuda.is_available():
                    status["gpu_device_name"] = torch.cuda.get_device_name(0)
                    status["gpu_memory_gb"] = torch.cuda.get_device_properties(0).total_memory / 1e9
            except:
                pass
    except:
        pass
    
    # Check XGBoost GPU support
    try:
        import xgboost as xgb
        # Create a small test model with GPU settings
        X_test = np.random.random((10, 10))
        y_test = np.random.randint(0, 2, 10)
        
        # First check with explicit GPU settings
        try:
            model = xgb.XGBClassifier(tree_method='gpu_hist', gpu_id=0)
            model.fit(X_test, y_test)
            status["xgboost_gpu"] = True
        except Exception as e:
            if "XGBoost is not compiled with CUDA support" in str(e):
                status["xgboost_gpu"] = False
            # Try with the newer API
            try:
                model = xgb.XGBClassifier(tree_method='hist', device='cuda')
                model.fit(X_test, y_test)
                status["xgboost_gpu"] = True
            except:
                status["xgboost_gpu"] = False
    except:
        pass
    
    # Check CatBoost GPU support
    try:
        from catboost import CatBoostClassifier
        try:
            model = CatBoostClassifier(task_type='GPU', devices='0', iterations=5, verbose=False)
            X_test = np.random.random((10, 10))
            y_test = np.random.randint(0, 2, 10)
            model.fit(X_test, y_test)
            status["catboost_gpu"] = True
        except Exception as e:
            if "GPU support was not enabled" in str(e):
                status["catboost_gpu"] = False
            else:
                # Try again with default settings
                try:
                    model = CatBoostClassifier(task_type='GPU', verbose=False)
                    model.fit(X_test, y_test)
                    status["catboost_gpu"] = True
                except:
                    status["catboost_gpu"] = False
    except:
        pass
    
    return status

In [7]:
def validate_dataframe(df, target_column):
    """
    Validate the DataFrame to ensure it's suitable for ML models
    
    Parameters:
    -----------
    df : pandas DataFrame
        DataFrame to validate
    target_column : str
        Name of the target column
        
    Returns:
    --------
    tuple: (is_valid, message)
    """
    # Check for missing values
    missing_values = df.isnull().sum()
    if missing_values.sum() > 0:
        return False, f"DataFrame contains {missing_values.sum()} missing values"
    
    # Check for non-numeric columns (except target which might be categorical)
    non_numeric_cols = [col for col in df.columns if col != target_column and 
                       not np.issubdtype(df[col].dtype, np.number)]
    if non_numeric_cols:
        return False, f"Non-numeric columns found: {non_numeric_cols}"
    
    # Check target column
    if not np.issubdtype(df[target_column].dtype, np.number):
        return False, f"Target column '{target_column}' is not numeric"
    
    return True, "DataFrame is valid"

In [10]:
"""
Modified train_pathogenicity_model function with ensemble SHAP support.

This replaces the function starting at line 1322 in the original notebook.
The key additions are:
1. Collection of SHAP values from base models
2. Averaging SHAP values for ensemble
3. Generating ensemble waterfall plots
4. Saving SHAP dataframes for all models
"""

# Define EnsembleExplainer at module level so it can be pickled
class EnsembleExplainer:
    """
    Mock explainer for ensemble that holds averaged expected values.
    Defined at module level so it can be pickled.
    """
    def __init__(self, expected_value):
        self.expected_value = expected_value


def train_pathogenicity_model(
    data_path,
    target_column="Oxygen2", 
    test_size=0.2,
    cv_folds=10,  # Only for "holdout" strategy
    random_state=42,
    output_dir="./model_output",
    binary_threshold=None,
    cv_strategy='loocv',
    n_jobs=-1,
    force_cpu=False
):
    """
    Train and evaluate pathogenicity prediction model using selected genes.
    NOW WITH ENSEMBLE SHAP SUPPORT!
    
    Parameters:
    -----------
    data_path : str
        Path to the selected genes dataset CSV file
    target_column : str
        Name of the column containing pathogenicity values
    test_size : float
        Proportion of data to use for testing (only used if cv_strategy='holdout')
    random_state : int
        Random seed for reproducibility
    output_dir : str
        Directory to save model and visualizations
    binary_threshold : int or None
        If provided, converts the problem to binary classification with this threshold
    cv_strategy : str
        Cross-validation strategy: 'loocv' for Leave-One-Out or 'holdout' for train/test split
    n_jobs : int
        Number of parallel jobs for cross-validation
        
    Returns:
    --------
    dict: Dictionary containing trained models and evaluation metrics
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Configure logging
    log_file = os.path.join(output_dir, "training_log.txt")
    def log_message(message):
        print(message)
        with open(log_file, 'a') as f:
            f.write(message + "\n")
    
    # Check GPU status
    if not force_cpu:
        log_message("Checking GPU availability...")
        gpu_status = check_gpu_status()
        
        log_message("GPU Status:")
        for key, value in gpu_status.items():
            log_message(f"  {key}: {value}")
        
        use_gpu_xgboost = gpu_status["xgboost_gpu"]
        use_gpu_catboost = gpu_status["catboost_gpu"]
        
        if not (use_gpu_xgboost or use_gpu_catboost):
            log_message("WARNING: No GPU support detected for any ML library. Using CPU.")
    else:
        log_message("Forcing CPU usage as requested.")
        use_gpu_xgboost = False
        use_gpu_catboost = False
    
    log_message(f"Loading data from {data_path}")
    
    # Properly load the data, setting index_col=0 to handle the unnamed index column
    df = pd.read_csv(data_path, index_col=0)
    
    # Ensuring that dataframe does not have names in one of the columns or something
    is_valid, message = validate_dataframe(df, target_column)
    if not is_valid:
        log_message(f"Error: {message}")
        log_message("Attempting to fix data types...")

        # Try to convert all columns to numeric
        for col in df.columns:
            if not np.issubdtype(df[col].dtype, np.number):
                try:
                    df[col] = pd.to_numeric(df[col])
                    log_message(f"Successfully converted {col} to numeric")
                except:
                    log_message(f"Failed to convert {col} to numeric")

        # Check again
        is_valid, message = validate_dataframe(df, target_column)
        if not is_valid:
            raise ValueError(message)
    
    # Verify the data was loaded correctly
    log_message(f"Data loaded: {df.shape[0]} samples, {df.shape[1]} columns")
    log_message(f"Column data types: {df.dtypes.value_counts().to_dict()}")
    
    # Separate features and target
    X = df.drop(columns=[target_column])
    y = df[target_column].values
    
    # Get feature names
    feature_names = X.columns.tolist()
    
    # Convert to binary classification if requested
    if binary_threshold is not None:
        log_message(f"Converting to binary classification with threshold {binary_threshold}")
        y_original = y.copy()
        y = (y >= binary_threshold).astype(int)
        class_names = ["Low Pathogenicity", "High Pathogenicity"]
        num_classes = 2
    else:
        # Get unique classes and their names
        unique_classes = sorted(np.unique(y))
        num_classes = len(unique_classes)
        class_names = [f"Class {cls}" for cls in unique_classes]
        log_message(f"Multi-class classification with {num_classes} classes: {unique_classes}")
    
    # Define base models
    base_models = [
        ('xgb', xgb.XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            tree_method='hist',  # Use 'hist' for both CPU and GPU
            device='cuda' if use_gpu_xgboost else 'cpu',  # New XGBoost API
            objective='multi:softprob' if num_classes > 2 else 'binary:logistic',
            num_class=num_classes if num_classes > 2 else None,
            random_state=random_state,
            enable_categorical=True  # Enable categorical feature support
        )),
        ('catboost', CatBoostClassifier(
            iterations=200,
            depth=5,
            learning_rate=0.05,
            loss_function='MultiClass' if num_classes > 2 else 'Logloss',
            task_type='GPU' if use_gpu_catboost else 'CPU',
            devices='0' if use_gpu_catboost else None,
            random_seed=random_state,
            verbose=False
        )),
        ('rf', RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            n_jobs=n_jobs,
            random_state=random_state
        ))
    ]
    
    # Create ensemble model
    ensemble = VotingClassifier(
        estimators=base_models,
        voting='soft'  # Use probability estimates for voting
    )
    
    # Store all models
    models = {name: model for name, model in base_models}
    models['ensemble'] = ensemble
    
    # Choose cross-validation strategy
    if cv_strategy == 'loocv':
        log_message("Using Leave-One-Out Cross-Validation")
        cv = LeaveOneOut()
        
        # Dictionary to store all results
        cv_results = {}
        all_probabilities = {}
        all_explanations = {}
        
        # Function to get explanations for each prediction
        def get_explanations(model, X_test_sample, model_name):
            """Get SHAP explanations for a single test sample"""
            # Create explainer
            explainer = shap.TreeExplainer(model)
            
            # Get SHAP values
            shap_values = explainer.shap_values(X_test_sample)
            
            return {
                'shap_values': shap_values,
                'expected_value': explainer.expected_value
            }
        
        # Loop through all models
        for name, model in models.items():
            log_message(f"Performing LOOCV for {name.upper()}...")
            
            # Initialize arrays to store predictions and true values
            y_pred = np.zeros_like(y)
            y_prob = np.zeros((len(y), num_classes))
            explanations = []
            
            # Use tqdm for progress bar
            for train_idx, test_idx in tqdm(cv.split(X), total=len(X)):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]
                
                # Train the model
                model.fit(X_train, y_train)
                
                # Make prediction
                y_pred[test_idx] = model.predict(X_test)
                
                # Store probability
                y_prob[test_idx] = model.predict_proba(X_test)
                
                # Store explanation (only for individual models, not ensemble)
                if name != 'ensemble':
                    explanations.append(get_explanations(model, X_test, name))
            
            # Calculate metrics
            accuracy = accuracy_score(y, y_pred)
            f1_macro = f1_score(y, y_pred, average='macro')
            f1_weighted = f1_score(y, y_pred, average='weighted')
            
            log_message(f"{name.upper()} - LOOCV Results:")
            log_message(f"  Accuracy: {accuracy:.4f}")
            log_message(f"  F1 Macro: {f1_macro:.4f}")
            log_message(f"  F1 Weighted: {f1_weighted:.4f}")
            
            # Store results
            cv_results[name] = {
                'accuracy': accuracy,
                'f1_macro': f1_macro,
                'f1_weighted': f1_weighted,
                'predictions': y_pred
            }
            
            all_probabilities[name] = y_prob
            
            if name != 'ensemble':
                all_explanations[name] = explanations
            
            # Create confusion matrix
            plt.figure(figsize=(10, 8))
            cm = confusion_matrix(y, y_pred)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=class_names, yticklabels=class_names)
            plt.xlabel('Predicted')
            plt.ylabel('True')
            plt.title(f'{name.upper()} - LOOCV Confusion Matrix')
            plt.tight_layout()
            plt.savefig(f"{output_dir}/{name}_loocv_confusion_matrix.png", dpi=200)
            plt.close()
            
            # Save classification report
            report = classification_report(y, y_pred, target_names=class_names)
            log_message(f"\nClassification Report for {name.upper()}:\n{report}")
            
            with open(f"{output_dir}/{name}_classification_report.txt", 'w') as f:
                f.write(report)
        
        # Now train final models on the entire dataset
        log_message("Training final models on the entire dataset...")
        for name, model in models.items():
            model.fit(X, y)
        
        # Create SHAP explainers for the final models
        explainers = {}
        for name, model in models.items():
            if name != 'ensemble':  # Only create explainers for base models
                log_message(f"Creating SHAP explainer for {name.upper()}...")
                explainers[name] = shap.TreeExplainer(model)
        
        # Store all relevant data for future use
        results = {
            'cv_results': cv_results,
            'all_probabilities': all_probabilities,
            'all_explanations': all_explanations,
            'models': models,
            'explainers': explainers,
            'feature_names': feature_names,
            'class_names': class_names,
            'num_classes': num_classes,
            'binary_threshold': binary_threshold
        }
        
    else:  # Holdout validation strategy with cross-validation
        log_message(f"Using holdout validation with {cv_folds}-fold cross-validation on training set")
        
        # Split data into train and test sets
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        
        log_message(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
        log_message(f"Class distribution in training: {np.bincount(y_train, minlength=num_classes)}")
        log_message(f"Class distribution in testing: {np.bincount(y_test, minlength=num_classes)}")
        
        # Set up cross-validation on training set
        from sklearn.model_selection import StratifiedKFold
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
        
        # Train and evaluate each model
        cv_results = {}
        all_probabilities = {}
        explainers = {}
        
        for name, model in models.items():
            log_message(f"Training {name.upper()} with {cv_folds}-fold CV...")
            
            # Perform cross-validation on training set
            cv_pred_labels = cross_val_predict(
                model, X_train, y_train, cv=cv, n_jobs=n_jobs
            )
            cv_predictions = cross_val_predict(
                model, X_train, y_train, cv=cv, method='predict_proba', n_jobs=n_jobs
            )
            
            # Calculate CV metrics on training set
            cv_accuracy = accuracy_score(y_train, cv_pred_labels)
            cv_f1_macro = f1_score(y_train, cv_pred_labels, average='macro', zero_division=0)
            cv_f1_weighted = f1_score(y_train, cv_pred_labels, average='weighted', zero_division=0)
            
            log_message(f"{name.upper()} - {cv_folds}-Fold CV Results on Training Set:")
            log_message(f"  CV Accuracy: {cv_accuracy:.4f}")
            log_message(f"  CV F1 Macro: {cv_f1_macro:.4f}")
            log_message(f"  CV F1 Weighted: {cv_f1_weighted:.4f}")
            
            # Train final model on full training set
            model.fit(X_train, y_train)
            
            # Make predictions on held-out test set
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)
            
            # Ensure y_pred is 1D array of class labels
            if y_pred.ndim > 1:
                y_pred = np.argmax(y_pred, axis=1)
            y_pred = y_pred.astype(int)
            
            # Calculate test set metrics
            test_accuracy = accuracy_score(y_test, y_pred)
            test_f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
            test_f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
            
            log_message(f"{name.upper()} - Final Test Results:")
            log_message(f"  Test Accuracy: {test_accuracy:.4f}")
            log_message(f"  Test F1 Macro: {test_f1_macro:.4f}")
            log_message(f"  Test F1 Weighted: {test_f1_weighted:.4f}")
            
            # Store results (using test set metrics as primary)
            cv_results[name] = {
                'cv_accuracy': cv_accuracy,
                'cv_f1_macro': cv_f1_macro,
                'cv_f1_weighted': cv_f1_weighted,
                'test_accuracy': test_accuracy,
                'test_f1_macro': test_f1_macro,
                'test_f1_weighted': test_f1_weighted,
                'accuracy': test_accuracy,  # For compatibility with plotting
                'f1_macro': test_f1_macro,
                'f1_weighted': test_f1_weighted,
                'predictions': y_pred
            }
            
            all_probabilities[name] = y_prob
            
            # Create confusion matrix for test set
            plt.figure(figsize=(10, 8))
            cm = confusion_matrix(y_test, y_pred)
            
            # Handle missing classes in test set
            # Ensure y_pred is 1D
            y_pred_1d = y_pred.flatten() if y_pred.ndim > 1 else y_pred
            present_classes = sorted(np.unique(np.concatenate([y_test, y_pred_1d])))
            present_class_names = [class_names[i] for i in present_classes]
            
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=present_class_names, 
                        yticklabels=present_class_names)
            plt.xlabel('Predicted')
            plt.ylabel('True')
            plt.title(f'{name.upper()} - Test Set Confusion Matrix')
            plt.tight_layout()
            plt.savefig(f"{output_dir}/{name}_test_confusion_matrix.png", dpi=200)
            plt.close()
            
            # Save classification report (with proper class handling)
            report = classification_report(
                y_test, 
                y_pred, 
                labels=present_classes,
                target_names=present_class_names,
                zero_division=0
            )
            log_message(f"\nClassification Report for {name.upper()}:\n{report}")
            
            with open(f"{output_dir}/{name}_classification_report.txt", 'w') as f:
                f.write(report)
            
            # Create SHAP explainer (only for base models)
            if name != 'ensemble':
                log_message(f"Creating SHAP explainer for {name.upper()}...")
                explainers[name] = shap.TreeExplainer(model)
                
                # Generate SHAP summary plot
                plt.figure(figsize=(12, 10))
                shap_values = explainers[name].shap_values(X_test)
                
                if num_classes == 2:
                    # For binary classification
                    if isinstance(shap_values, list):
                        # If shap_values is a list, use the second element (positive class)
                        shap.summary_plot(shap_values[1], X_test, feature_names=feature_names, show=False)
                    else:
                        # Otherwise use as is
                        shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
                else:
                    # For multi-class, create a summary for all classes combined
                    shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
                
                plt.title(f"{name.upper()} - SHAP Feature Importance")
                plt.tight_layout()
                plt.savefig(f"{output_dir}/{name}_shap_summary.png", dpi=200)
                plt.close()
                
                # Generate some sample waterfall plots
                # Choose a few test samples to explain
                log_message(f"Generating sample waterfall plots for {name.upper()}...")
                num_samples = min(5, len(X_test))
                for i in range(num_samples):
                    generate_waterfall_plot(
                        X_test.iloc[i], 
                        y_test[i], 
                        i, 
                        explainers[name], 
                        shap_values, 
                        feature_names, 
                        class_names, 
                        os.path.join(output_dir, name), 
                        num_classes
                    )
        
        # =====================================================================
        # NEW CODE: Create ensemble SHAP values by averaging base models
        # =====================================================================
        log_message("Creating ensemble SHAP explainer by averaging base models...")
        
        # Collect SHAP values from all base models
        all_shap_values = {}
        all_base_values = {}
        
        for name in ['xgb', 'catboost', 'rf']:
            if name in explainers:
                shap_vals = explainers[name].shap_values(X_test)
                all_shap_values[name] = shap_vals
                all_base_values[name] = explainers[name].expected_value
        
        # Average SHAP values across models
        if len(all_shap_values) > 0:
            # Handle binary vs multi-class
            if num_classes == 2:
                # For binary classification, check if values are in list format
                ensemble_shap = []
                ensemble_base = []
                
                for class_idx in range(2):
                    class_shap_values = []
                    class_base_values = []
                    
                    for name in ['xgb', 'catboost', 'rf']:
                        if name in all_shap_values:
                            shap_vals = all_shap_values[name]
                            base_vals = all_base_values[name]
                            
                            # Handle different formats
                            if isinstance(shap_vals, list):
                                class_shap_values.append(shap_vals[class_idx])
                                if isinstance(base_vals, (list, np.ndarray)) and len(base_vals) > 1:
                                    class_base_values.append(base_vals[class_idx])
                                else:
                                    # Single value - use it for positive class, inverse for negative
                                    if class_idx == 1:
                                        class_base_values.append(base_vals if not isinstance(base_vals, (list, np.ndarray)) else base_vals[0])
                                    else:
                                        class_base_values.append(-(base_vals if not isinstance(base_vals, (list, np.ndarray)) else base_vals[0]))
                            else:
                                # Single array format (usually for positive class only)
                                if class_idx == 1:
                                    class_shap_values.append(shap_vals)
                                    class_base_values.append(base_vals if not isinstance(base_vals, (list, np.ndarray)) else base_vals[0])
                                else:
                                    class_shap_values.append(-shap_vals)
                                    class_base_values.append(-(base_vals if not isinstance(base_vals, (list, np.ndarray)) else base_vals[0]))
                    
                    # Average across models
                    if len(class_shap_values) > 0:
                        ensemble_shap.append(np.mean(class_shap_values, axis=0))
                        ensemble_base.append(np.mean(class_base_values))
                
                ensemble_shap_values = ensemble_shap
                # Store as scalar for class 1 (positive class) for waterfall plots
                ensemble_base_value = float(ensemble_base[1])  # Convert to scalar
                ensemble_base_value_full = np.array(ensemble_base)  # Keep full array for reference
            else:
                # Multi-class
                ensemble_shap_values = []
                ensemble_base_value = []
                
                for class_idx in range(num_classes):
                    class_shap = []
                    class_base = []
                    
                    for name in ['xgb', 'catboost', 'rf']:
                        if name in all_shap_values:
                            class_shap.append(all_shap_values[name][class_idx])
                            class_base.append(all_base_values[name][class_idx])
                    
                    ensemble_shap_values.append(np.mean(class_shap, axis=0))
                    ensemble_base_value.append(np.mean(class_base))
                
                ensemble_base_value = np.array(ensemble_base_value)
            
            # Create a mock explainer object for ensemble with averaged values
            # Use the module-level EnsembleExplainer class (defined above)
            explainers['ensemble'] = EnsembleExplainer(ensemble_base_value)
            
            # Generate ensemble SHAP summary plot
            log_message("Generating ensemble SHAP summary plot...")
            plt.figure(figsize=(12, 10))
            if num_classes == 2:
                # For binary classification, use positive class
                shap.summary_plot(ensemble_shap_values[1], X_test, feature_names=feature_names, show=False)
            else:
                # For multi-class
                shap.summary_plot(ensemble_shap_values, X_test, feature_names=feature_names, show=False)
            plt.title("ENSEMBLE - SHAP Feature Importance")
            plt.tight_layout()
            plt.savefig(f"{output_dir}/ensemble_shap_summary.png", dpi=200)
            plt.close()
            
            # Generate ensemble waterfall plots
            log_message("Generating ensemble waterfall plots...")
            num_samples = min(5, len(X_test))
            for i in range(num_samples):
                generate_waterfall_plot(
                    X_test.iloc[i], 
                    y_test[i], 
                    i, 
                    explainers['ensemble'], 
                    ensemble_shap_values, 
                    feature_names, 
                    class_names, 
                    os.path.join(output_dir, 'ensemble'), 
                    num_classes
                )
            
            # Save SHAP values as DataFrames for all models including ensemble
            log_message("Saving SHAP values dataframes...")
            shap_dfs_dir = os.path.join(output_dir, "shap_dataframes")
            os.makedirs(shap_dfs_dir, exist_ok=True)
            
            for model_name in ['xgb', 'catboost', 'rf', 'ensemble']:
                if model_name == 'ensemble':
                    shap_vals = ensemble_shap_values
                elif model_name in all_shap_values:
                    shap_vals = all_shap_values[model_name]
                else:
                    continue
                
                # Handle different SHAP value formats
                if isinstance(shap_vals, list):
                    # Multi-class or binary with separate arrays per class
                    if num_classes == 2:
                        # For binary classification, only save the positive class (class 1)
                        class_idx = 1
                        class_name = class_names[class_idx]
                        df = pd.DataFrame(
                            shap_vals[class_idx],
                            columns=feature_names,
                            index=X_test.index
                        )
                        # Add metadata columns
                        df.insert(0, 'true_class', y_test)
                        df.insert(0, 'sample_idx', range(len(X_test)))
                        
                        filename = f"{model_name}_shap_values_{class_name.replace(' ', '_')}.csv"
                        df.to_csv(os.path.join(shap_dfs_dir, filename), index=True)
                        log_message(f"  Saved {filename}")
                    else:
                        # For multi-class, save all classes
                        for class_idx, class_name in enumerate(class_names):
                            df = pd.DataFrame(
                                shap_vals[class_idx],
                                columns=feature_names,
                                index=X_test.index
                            )
                            # Add metadata columns
                            df.insert(0, 'true_class', y_test)
                            df.insert(0, 'sample_idx', range(len(X_test)))
                            
                            filename = f"{model_name}_shap_values_class_{class_idx}_{class_name.replace(' ', '_')}.csv"
                            df.to_csv(os.path.join(shap_dfs_dir, filename), index=True)
                            log_message(f"  Saved {filename}")
                else:
                    # Single array (binary classification, positive class only)
                    df = pd.DataFrame(
                        shap_vals,
                        columns=feature_names,
                        index=X_test.index
                    )
                    # Add metadata columns
                    df.insert(0, 'true_class', y_test)
                    df.insert(0, 'sample_idx', range(len(X_test)))
                    
                    filename = f"{model_name}_shap_values.csv"
                    df.to_csv(os.path.join(shap_dfs_dir, filename), index=True)
                    log_message(f"  Saved {filename}")
        # =====================================================================
        # END NEW CODE
        # =====================================================================
        
        # Store all relevant data for future use
        results = {
            'cv_results': cv_results,
            'all_probabilities': all_probabilities,
            'models': models,
            'explainers': explainers,
            'feature_names': feature_names,
            'class_names': class_names,
            'num_classes': num_classes,
            'binary_threshold': binary_threshold,
            'X_test': X_test,
            'y_test': y_test
        }
    
    # Save models for future use
    log_message("Saving models and results...")
    with open(f"{output_dir}/trained_models.pkl", 'wb') as f:
        pickle.dump(models, f)
    
    # Save explainers
    with open(f"{output_dir}/shap_explainers.pkl", 'wb') as f:
        pickle.dump(explainers, f)
    
    # Save feature names
    with open(f"{output_dir}/feature_names.pkl", 'wb') as f:
        pickle.dump(feature_names, f)
    
    # Save class information
    class_info = {
        'class_names': class_names,
        'num_classes': num_classes,
        'binary_threshold': binary_threshold
    }
    with open(f"{output_dir}/class_info.pkl", 'wb') as f:
        pickle.dump(class_info, f)
    
    # Save results
    with open(f"{output_dir}/results.pkl", 'wb') as f:
        pickle.dump(results, f)
    
    # Create summary visualization of model performance
    create_model_comparison_plot(cv_results, output_dir)
    
    log_message(f"Model training and evaluation complete! Results saved to {output_dir}")
    
    return results

def generate_waterfall_plot(
    sample, 
    true_class, 
    sample_idx, 
    explainer, 
    shap_values, 
    feature_names, 
    class_names, 
    output_dir, 
    num_classes
):
    """
    Generate waterfall plot for a specific sample with extensive debugging
    
    Parameters:
    -----------
    sample : pandas Series
        Single sample to explain
    true_class : int
        Actual class of the sample
    sample_idx : int
        Index of the sample in the dataset
    explainer : shap.TreeExplainer
        SHAP explainer object
    shap_values : list or ndarray
        Precomputed SHAP values
    feature_names : list
        List of feature names
    class_names : list
        List of class names
    output_dir : str
        Directory to save plots
    num_classes : int
        Number of classes
    """
    print(f"\n=== DEBUG: Waterfall plot for sample {sample_idx} ===")
    print(f"True class: {true_class} ({class_names[true_class]})")
    print(f"Number of classes: {num_classes}")
    print(f"Sample shape: {sample.shape}")
    print(f"Number of features: {len(feature_names)}")
    
    # Debug SHAP values structure
    print(f"\n--- SHAP Values Debug ---")
    print(f"SHAP values type: {type(shap_values)}")
    print(f"Is list: {isinstance(shap_values, list)}")
    
    if isinstance(shap_values, list):
        print(f"List length: {len(shap_values)}")
        for i, sv in enumerate(shap_values):
            print(f"  shap_values[{i}] shape: {sv.shape}")
            print(f"  shap_values[{i}] type: {type(sv)}")
    else:
        print(f"Array shape: {shap_values.shape}")
        print(f"Array ndim: {shap_values.ndim}")
        print(f"Array dtype: {shap_values.dtype}")
    
    # Debug explainer expected value
    print(f"\n--- Explainer Debug ---")
    print(f"Expected value type: {type(explainer.expected_value)}")
    if hasattr(explainer.expected_value, '__len__'):
        print(f"Expected value length: {len(explainer.expected_value)}")
        print(f"Expected values: {explainer.expected_value}")
    else:
        print(f"Expected value: {explainer.expected_value}")
    
    # Create a directory for waterfall plots if it doesn't exist
    waterfall_dir = os.path.join(output_dir, "waterfall_plots")
    os.makedirs(waterfall_dir, exist_ok=True)
    
    if num_classes == 2:
        print(f"\n--- Binary Classification Path ---")
        
        # Handle different shap_values formats
        if isinstance(shap_values, list):
            print("Using list format...")
            print(f"List has {len(shap_values)} elements")
            
            if len(shap_values) == 1:
                # Some binary classifiers return a single element list
                print("Single element list detected")
                values = shap_values[0][sample_idx]
                base_value = explainer.expected_value[0] if isinstance(explainer.expected_value, list) else explainer.expected_value
                print(f"Using shap_values[0][{sample_idx}]")
            else:
                # Standard binary with two classes
                print("Two element list detected - using positive class (index 1)")
                values = shap_values[1][sample_idx]
                if isinstance(explainer.expected_value, (list, np.ndarray)) and len(explainer.expected_value) > 1:
                    base_value = float(explainer.expected_value[1])
                else:
                    base_value = float(explainer.expected_value)
                print(f"Using shap_values[1][{sample_idx}]")
        else:
            print("Using array format...")
            if shap_values.ndim == 3:
                print(f"3D array detected: {shap_values.shape}")
                print("Using positive class (class 1) for binary classification")
                values = shap_values[sample_idx, :, 1]  # Use positive class
                base_value = explainer.expected_value[1] if hasattr(explainer.expected_value, '__len__') else explainer.expected_value
                print(f"Using shap_values[{sample_idx}, :, 1]")
            elif shap_values.ndim == 2:
                print(f"2D array detected: {shap_values.shape}")
                values = shap_values[sample_idx]
                base_value = explainer.expected_value
                print(f"Using shap_values[{sample_idx}]")
            else:
                print(f"Unexpected array dimension: {shap_values.ndim}")
                raise ValueError(f"Unexpected SHAP values dimension: {shap_values.ndim}")
        
        print(f"Final values shape: {values.shape}")
        print(f"Final values type: {type(values)}")
        print(f"Base value: {base_value}")
        print(f"Values sample (first 5): {values[:5]}")
        
        try:
            plt.figure(figsize=(12, 8))
            
            # Create the explanation object for a single sample
            explanation = shap.Explanation(
                values=values, 
                base_values=base_value, 
                data=sample.values,
                feature_names=feature_names
            )
            
            print(f"Explanation created successfully")
            print(f"Explanation values shape: {explanation.values.shape}")
            print(f"Explanation data shape: {explanation.data.shape}")
            
            # Generate the waterfall plot
            shap.waterfall_plot(explanation, show=False)
            
            plt.title(f"Sample {sample_idx} - True Class: {class_names[true_class]}")
            plt.tight_layout()
            plt.savefig(f"{waterfall_dir}/waterfall_sample_{sample_idx}.png", dpi=200)
            plt.close()
            print(f"Waterfall plot saved successfully")
            
        except Exception as e:
            print(f"ERROR in waterfall plot generation: {e}")
            print(f"Error type: {type(e)}")
            import traceback
            traceback.print_exc()
            plt.close()  # Make sure to close the figure
        
    else:
        print(f"\n--- Multi-class Classification Path ---")
        # For multi-class, create a waterfall plot for each class
        for class_idx in range(num_classes):
            print(f"\n--- Processing class {class_idx} ({class_names[class_idx]}) ---")
            
            # Extract SHAP values for this specific class and sample
            if isinstance(shap_values, list):
                print("Using list format...")
                values = shap_values[class_idx][sample_idx]
                base_value = explainer.expected_value[class_idx]
                print(f"Using shap_values[{class_idx}][{sample_idx}]")
            else:
                print("Using array format...")
                if shap_values.ndim == 3:
                    print(f"3D array detected: {shap_values.shape}")
                    values = shap_values[sample_idx, :, class_idx]
                    base_value = explainer.expected_value[class_idx] if hasattr(explainer.expected_value, '__len__') else explainer.expected_value
                    print(f"Using shap_values[{sample_idx}, :, {class_idx}]")
                elif shap_values.ndim == 2:
                    print(f"2D array detected: {shap_values.shape}")
                    if class_idx == 0:  # Only plot for the first class in 2D case
                        values = shap_values[sample_idx]
                        base_value = explainer.expected_value
                        print(f"Using shap_values[{sample_idx}] for 2D case")
                    else:
                        print(f"Skipping class {class_idx} for 2D array")
                        continue
                else:
                    print(f"Unexpected array dimension: {shap_values.ndim}")
                    raise ValueError(f"Unexpected SHAP values dimension: {shap_values.ndim}")
            
            print(f"Final values shape: {values.shape}")
            print(f"Final values type: {type(values)}")
            print(f"Base value: {base_value}")
            print(f"Values sample (first 5): {values[:5]}")
            
            try:
                plt.figure(figsize=(12, 8))
                
                # Create the explanation object for a single sample and class
                explanation = shap.Explanation(
                    values=values, 
                    base_values=base_value, 
                    data=sample.values,
                    feature_names=feature_names
                )
                
                print(f"Explanation created successfully")
                print(f"Explanation values shape: {explanation.values.shape}")
                print(f"Explanation data shape: {explanation.data.shape}")
                
                # Generate the waterfall plot
                shap.waterfall_plot(explanation, show=False)
                
                plt.title(f"Sample {sample_idx} - True Class: {class_names[true_class]} - Impact on {class_names[class_idx]}")
                plt.tight_layout()
                plt.savefig(f"{waterfall_dir}/waterfall_sample_{sample_idx}_class_{class_idx}.png", dpi=200)
                plt.close()
                print(f"Waterfall plot for class {class_idx} saved successfully")
                
            except Exception as e:
                print(f"ERROR in waterfall plot generation for class {class_idx}: {e}")
                print(f"Error type: {type(e)}")
                import traceback
                traceback.print_exc()
                plt.close()  # Make sure to close the figure
    
    print(f"=== END DEBUG for sample {sample_idx} ===\n")

def create_model_comparison_plot(cv_results, output_dir):
    """
    Create a visualization comparing model performance
    
    Parameters:
    -----------
    cv_results : dict
        Dictionary containing model results
    output_dir : str
        Directory to save the plot
    """
    # Extract metrics for each model
    models = list(cv_results.keys())
    accuracy = [cv_results[model]['accuracy'] for model in models]
    f1_macro = [cv_results[model]['f1_macro'] for model in models]
    f1_weighted = [cv_results[model]['f1_weighted'] for model in models]
    
    # Create a dataframe for plotting
    df = pd.DataFrame({
        'Model': models * 3,
        'Metric': ['Accuracy'] * len(models) + ['F1 Macro'] * len(models) + ['F1 Weighted'] * len(models),
        'Score': accuracy + f1_macro + f1_weighted
    })
    
    # Create the plot
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Model', y='Score', hue='Metric', data=df)
    plt.title('Model Performance Comparison')
    plt.ylim(0, 1.0)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/model_comparison.png", dpi=200)
    plt.close()

def is_cuda_available():
    """Check if CUDA is available for GPU acceleration"""
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        try:
            import tensorflow as tf
            return tf.test.is_gpu_available(cuda_only=True)
        except:
            return False

In [11]:
def predict_new_genome(
    genome_data, 
    model_dir="./model_output", 
    model_type='ensemble',
    output_dir=None
):
    """
    Predict pathogenicity for a new genome and explain the prediction
    
    Parameters:
    -----------
    genome_data : pandas DataFrame or path to CSV
        New genome data as DataFrame or path to CSV file
    model_dir : str
        Directory containing the saved model and related files
    model_type : str
        Type of model to use ('xgb', 'catboost', 'rf', or 'ensemble')
    output_dir : str or None
        Directory to save visualizations. If None, uses model_dir
        
    Returns:
    --------
    dict: Dictionary containing prediction and explanation
    """
    if output_dir is None:
        output_dir = os.path.join(model_dir, "new_predictions")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Load the data if it's a path
    if isinstance(genome_data, str):
        # Handle the index column properly
        genome_data = pd.read_csv(genome_data, index_col=0)
    
    # Load the models
    with open(f"{model_dir}/trained_models.pkl", 'rb') as f:
        models = pickle.load(f)
    
    # Load the explainers
    with open(f"{model_dir}/shap_explainers.pkl", 'rb') as f:
        explainers = pickle.load(f)
    
    # Load feature names
    with open(f"{model_dir}/feature_names.pkl", 'rb') as f:
        feature_names = pickle.load(f)
    
    # Load class information
    with open(f"{model_dir}/class_info.pkl", 'rb') as f:
        class_info = pickle.load(f)
    
    class_names = class_info['class_names']
    num_classes = class_info['num_classes']
    
    # Select the model
    if model_type not in models:
        raise ValueError(f"Model type '{model_type}' not found. Available models: {list(models.keys())}")
    
    model = models[model_type]
    
    # Select explainer - use XGB explainer for ensemble, otherwise use the model-specific explainer
    if model_type == 'ensemble':
        explainer = explainers['xgb']  # Use XGB explainer for ensemble
        shap_model = models['xgb']     # Use XGB model for SHAP
    else:
        explainer = explainers[model_type]
        shap_model = models[model_type]
    
    # Ensure the genome data has all the required features
    missing_features = [feat for feat in feature_names if feat not in genome_data.columns]
    if missing_features:
        print(f"Warning: Missing {len(missing_features)} features in the genome data")
        # Fill missing features with zeros
        for feat in missing_features:
            genome_data[feat] = 0
    
    # Select only the features used by the model
    X = genome_data[feature_names]
    
    # Make prediction
    prediction = model.predict(X)[0]
    
    # Get probability distribution
    probabilities = model.predict_proba(X)[0]
    
    print(f"Prediction: {class_names[prediction]}")
    print(f"Probability distribution: {dict(zip(class_names, probabilities))}")
    
    # Calculate SHAP values
    shap_values = explainer.shap_values(X)
    
    # Create waterfall plots for all samples
    for i in range(len(X)):
        generate_waterfall_plot(
            X.iloc[i], 
            prediction,  # Using prediction as "true" class since we don't know the actual class
            i, 
            explainer, 
            shap_values, 
            feature_names, 
            class_names, 
            output_dir, 
            num_classes
        )
    
    # Save prediction results
    prediction_results = {
        'prediction': prediction,
        'class_name': class_names[prediction],
        'probabilities': dict(zip(class_names, probabilities)),
        'shap_values': shap_values
    }
    
    with open(f"{output_dir}/prediction_results.pkl", 'wb') as f:
        pickle.dump(prediction_results, f)
    
    return prediction_results

In [12]:
def predict_new_genome_with_explanations(
    genome_data, 
    model_dir="./model_output", 
    model_type='ensemble',
    output_dir=None,
    include_gene_clusters=True,
    n_clusters=10
):
    """
    Predict pathogenicity for a new genome with detailed explanations and visualizations
    
    Parameters:
    -----------
    genome_data : pandas DataFrame or path to CSV
        New genome data as DataFrame or path to CSV file
    model_dir : str
        Directory containing the saved model and related files
    model_type : str
        Type of model to use ('xgb', 'catboost', 'rf', or 'ensemble')
    output_dir : str or None
        Directory to save visualizations. If None, uses model_dir/new_predictions
    include_gene_clusters : bool
        Whether to include gene cluster information in the explanation
    n_clusters : int
        Number of gene clusters to use (only if include_gene_clusters is True)
        
    Returns:
    --------
    dict: Dictionary containing prediction and explanation
    """
    if output_dir is None:
        output_dir = os.path.join(model_dir, "new_predictions")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Load the data if it's a path
    if isinstance(genome_data, str):
        # Handle the index column properly
        genome_data = pd.read_csv(genome_data, index_col=0)
    
    # Load the models
    with open(f"{model_dir}/trained_models.pkl", 'rb') as f:
        models = pickle.load(f)
    
    # Load the explainers
    with open(f"{model_dir}/shap_explainers.pkl", 'rb') as f:
        explainers = pickle.load(f)
    
    # Load feature names
    with open(f"{model_dir}/feature_names.pkl", 'rb') as f:
        feature_names = pickle.load(f)
    
    # Load class information
    with open(f"{model_dir}/class_info.pkl", 'rb') as f:
        class_info = pickle.load(f)
    
    class_names = class_info['class_names']
    num_classes = class_info['num_classes']
    
    # Select the model
    if model_type not in models:
        raise ValueError(f"Model type '{model_type}' not found. Available models: {list(models.keys())}")
    
    model = models[model_type]
    
    # Select explainer - use XGB explainer for ensemble, otherwise use the model-specific explainer
    if model_type == 'ensemble':
        explainer = explainers['xgb']  # Use XGB explainer for ensemble
        shap_model = models['xgb']     # Use XGB model for SHAP
    else:
        explainer = explainers[model_type]
        shap_model = models[model_type]
    
    # Ensure the genome data has all the required features
    missing_features = [feat for feat in feature_names if feat not in genome_data.columns]
    if missing_features:
        print(f"Warning: Missing {len(missing_features)} features in the genome data")
        # Fill missing features with zeros
        for feat in missing_features:
            genome_data[feat] = 0
    
    # Select only the features used by the model
    X = genome_data[feature_names]
    
    # Make prediction
    prediction = model.predict(X)[0]
    
    # Get probability distribution
    probabilities = model.predict_proba(X)[0]
    
    print(f"Prediction: {class_names[prediction]}")
    print(f"Probability distribution: {dict(zip(class_names, probabilities))}")
    
    # Calculate SHAP values
    shap_values = explainer.shap_values(X)
    
    # Create waterfall plots for all samples
    for i in range(len(X)):
        # Generate interactive waterfall plot
        generate_interactive_waterfall_plot(
            X.iloc[i], 
            prediction,  # Using prediction as "true" class since we don't know the actual class
            i, 
            explainer, 
            shap_values, 
            feature_names, 
            class_names, 
            output_dir, 
            num_classes
        )
    
    # Create probability distribution visualization
    prob_data = pd.DataFrame({
        'Class': class_names,
        'Probability': probabilities
    })
    
    prob_fig = px.bar(
        prob_data,
        x='Class',
        y='Probability',
        color='Probability',
        title=f"Predicted Probability Distribution (Predicted Class: {class_names[prediction]})",
        color_continuous_scale="Viridis"
    )
    
    prob_fig.update_layout(height=500, width=800)
    prob_fig.write_html(os.path.join(output_dir, "probability_distribution.html"))
    
    # Add gene cluster information if requested
    if include_gene_clusters:
        # Check if clustering results already exist
        clusters_dir = os.path.join(model_dir, "gene_clusters")
        cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
        
        if os.path.exists(cluster_file):
            with open(cluster_file, 'rb') as f:
                cluster_results = pickle.load(f)
        else:
            # Perform clustering
            cluster_results = analyze_gene_clusters(model_dir, n_clusters)
        
        if cluster_results:
            # Identify important clusters for this prediction
            feature_cluster_map = cluster_results['feature_cluster_map']
            
            # Analyze SHAP values by cluster
            cluster_impact = {}
            
            if isinstance(shap_values, list):  # Multi-class case
                for class_idx in range(num_classes):
                    cluster_impact[class_idx] = {}
                    
                    for cluster_id in range(n_clusters):
                        # Get features in this cluster
                        cluster_features = [f for f, c in feature_cluster_map.items() if c == cluster_id]
                        
                        # Get feature indices
                        feature_indices = [feature_names.index(f) for f in cluster_features if f in feature_names]
                        
                        if feature_indices:
                            # Calculate total and mean SHAP for this cluster
                            cluster_shap_total = shap_values[class_idx][0, feature_indices].sum()
                            cluster_shap_mean = shap_values[class_idx][0, feature_indices].mean()
                            
                            cluster_impact[class_idx][cluster_id] = {
                                'total_impact': cluster_shap_total,
                                'mean_impact': cluster_shap_mean,
                                'n_features': len(feature_indices)
                            }
            else:  # Binary classification case
                for cluster_id in range(n_clusters):
                    # Get features in this cluster
                    cluster_features = [f for f, c in feature_cluster_map.items() if c == cluster_id]
                    
                    # Get feature indices
                    feature_indices = [feature_names.index(f) for f in cluster_features if f in feature_names]
                    
                    if feature_indices:
                        # Calculate total and mean SHAP for this cluster
                        cluster_shap_total = shap_values[0, feature_indices].sum()
                        cluster_shap_mean = shap_values[0, feature_indices].mean()
                        
                        cluster_impact[0][cluster_id] = {
                            'total_impact': cluster_shap_total,
                            'mean_impact': cluster_shap_mean,
                            'n_features': len(feature_indices)
                        }
            
            # Create visualization of cluster impact
            if isinstance(shap_values, list):  # Multi-class case
                for class_idx in range(num_classes):
                    # Prepare data for visualization
                    data = []
                    for cluster_id, impact_data in cluster_impact[class_idx].items():
                        data.append({
                            'cluster_id': f"Cluster {cluster_id}",
                            'total_impact': impact_data['total_impact'],
                            'mean_impact': impact_data['mean_impact'],
                            'n_features': impact_data['n_features']
                        })
                    
                    imp_df = pd.DataFrame(data)
                    
                    # Create bar plot for total impact
                    fig_total = px.bar(
                        imp_df,
                        x='cluster_id',
                        y='total_impact',
                        color='n_features',
                        labels={'total_impact': 'Total SHAP Impact', 'cluster_id': 'Gene Cluster'},
                        title=f"Cluster Impact on Class {class_names[class_idx]} Prediction",
                        color_continuous_scale="Viridis"
                    )
                    
                    fig_total.update_layout(height=500, width=800)
                    fig_total.write_html(os.path.join(output_dir, f"cluster_impact_class_{class_idx}.html"))
                    
                    # Create bar plot for mean impact
                    fig_mean = px.bar(
                        imp_df,
                        x='cluster_id',
                        y='mean_impact',
                        color='n_features',
                        labels={'mean_impact': 'Mean SHAP Impact', 'cluster_id': 'Gene Cluster'},
                        title=f"Average Gene Impact by Cluster for Class {class_names[class_idx]}",
                        color_continuous_scale="Viridis"
                    )
                    
                    fig_mean.update_layout(height=500, width=800)
                    fig_mean.write_html(os.path.join(output_dir, f"cluster_mean_impact_class_{class_idx}.html"))
            else:  # Binary classification case
                # Prepare data for visualization
                data = []
                for cluster_id, impact_data in cluster_impact[0].items():
                    data.append({
                        'cluster_id': f"Cluster {cluster_id}",
                        'total_impact': impact_data['total_impact'],
                        'mean_impact': impact_data['mean_impact'],
                        'n_features': impact_data['n_features']
                    })
                
                imp_df = pd.DataFrame(data)
                
                # Create bar plot for total impact
                fig_total = px.bar(
                    imp_df,
                    x='cluster_id',
                    y='total_impact',
                    color='n_features',
                    labels={'total_impact': 'Total SHAP Impact', 'cluster_id': 'Gene Cluster'},
                    title="Cluster Impact on Prediction",
                    color_continuous_scale="Viridis"
                )
                
                fig_total.update_layout(height=500, width=800)
                fig_total.write_html(os.path.join(output_dir, "cluster_impact.html"))
                
                # Create bar plot for mean impact
                fig_mean = px.bar(
                    imp_df,
                    x='cluster_id',
                    y='mean_impact',
                    color='n_features',
                    labels={'mean_impact': 'Mean SHAP Impact', 'cluster_id': 'Gene Cluster'},
                    title="Average Gene Impact by Cluster",
                    color_continuous_scale="Viridis"
                )
                
                fig_mean.update_layout(height=500, width=800)
                fig_mean.write_html(os.path.join(output_dir, "cluster_mean_impact.html"))
    
    # Save prediction results
    prediction_results = {
        'prediction': prediction,
        'class_name': class_names[prediction],
        'probabilities': dict(zip(class_names, probabilities)),
        'shap_values': shap_values
    }
    
    # Add cluster impact if calculated
    if include_gene_clusters and 'cluster_impact' in locals():
        prediction_results['cluster_impact'] = cluster_impact
    
    with open(f"{output_dir}/prediction_results.pkl", 'wb') as f:
        pickle.dump(prediction_results, f)
    
    print(f"Prediction and explanations saved to {output_dir}")
    
    return prediction_results

## New Method of training and saving model

In [13]:
from model_pipeline import train_and_save_model

results, package_path = train_and_save_model(
    train_function=train_pathogenicity_model,
    data_path="./data/selected_genes_dataset.csv",
    output_dir="./models",
    package_name="oxygen_classifier",
    target_column="Oxygen2",
    cv_strategy='holdout'
)

'''
# Alternate option not using the wrapper function

from model_pipeline import save_model_package, train_and_save_model

# Keep your existing training call AND THEN save it
results = train_pathogenicity_model(
    data_path="./data/oxygen_data.csv",
    output_dir="./models",
    target_column="Oxygen2",
    cv_strategy='holdout'
)

# Now save as a package
package_path = save_model_package(results, "./models", package_name="oxygen_classifier")
'''

Training model...
Checking GPU availability...
GPU Status:
  xgboost_gpu: True
  catboost_gpu: False
  system_cuda: False
  gpu_device_name: None
  gpu_memory_gb: None
Loading data from ./data/selected_genes_dataset.csv
Data loaded: 5520 samples, 21 columns
Column data types: {dtype('int64'): 21}
Multi-class classification with 2 classes: [0, 1]
Using holdout validation with 10-fold cross-validation on training set
Training set: (4416, 20), Test set: (1104, 20)
Class distribution in training: [3382 1034]
Class distribution in testing: [845 259]
Training XGB with 10-fold CV...
XGB - 10-Fold CV Results on Training Set:
  CV Accuracy: 0.9676
  CV F1 Macro: 0.9546
  CV F1 Weighted: 0.9675
XGB - Final Test Results:
  Test Accuracy: 0.9583
  Test F1 Macro: 0.9412
  Test F1 Weighted: 0.9580

Classification Report for XGB:
              precision    recall  f1-score   support

     Class 0       0.97      0.98      0.97       845
     Class 1       0.93      0.89      0.91       259

    accur

'\n# Alternate option not using the wrapper function\n\nfrom model_pipeline import save_model_package, train_and_save_model\n\n# Keep your existing training call AND THEN save it\nresults = train_pathogenicity_model(\n    data_path="./data/oxygen_data.csv",\n    output_dir="./models",\n    target_column="Oxygen2",\n    cv_strategy=\'holdout\'\n)\n\n# Now save as a package\npackage_path = save_model_package(results, "./models", package_name="oxygen_classifier")\n'

/usr/local/lib/python3.8/site-packages/xgboost/core.py:158: UserWarning: [22:36:00] WARNING: /workspace/src/context.cc:43: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.8/site-packages/xgboost/core.py:158: UserWarning: [22:36:00] WARNING: /workspace/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.8/site-packages/xgboost/core.py:158: UserWarning: [22:36:43] WARNING: /workspace/src/context.cc:43: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.8/site-packages/xgboost/core.py:158: UserWarning: [22:36:43] WARNING: /workspace/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.8/site-packages/xgboost/core.py:158: UserWarning: [22:35:59] WARNING: /workspace/src/context.cc:43: No visible GPU is found, setting device to CPU.
  warnings.wa

## Old Method of training and saving model

In [12]:
# Train model
results = train_pathogenicity_model(
    data_path="./data/elias_all_phyla_selected_genes_dataset_20.csv",
    output_dir="oxygen_model",
    cv_folds = 10,
    cv_strategy='holdout', # 'loocv' (recommended for small datasets), 'holdout'
    force_cpu=True  # Set to True if you want to force CPU usage
)

Forcing CPU usage as requested.
Loading data from ./data/elias_all_phyla_selected_genes_dataset_20.csv
Data loaded: 5520 samples, 21 columns
Column data types: {dtype('int64'): 21}
Multi-class classification with 2 classes: [0, 1]
Using holdout validation with 10-fold cross-validation on training set
Training set: (4416, 20), Test set: (1104, 20)
Class distribution in training: [3382 1034]
Class distribution in testing: [845 259]
Training XGB with 10-fold CV...
XGB - 10-Fold CV Results on Training Set:
  CV Accuracy: 0.9667
  CV F1 Macro: 0.9534
  CV F1 Weighted: 0.9666
XGB - Final Test Results:
  Test Accuracy: 0.9574
  Test F1 Macro: 0.9405
  Test F1 Weighted: 0.9573

Classification Report for XGB:
              precision    recall  f1-score   support

     Class 0       0.97      0.97      0.97       845
     Class 1       0.91      0.90      0.91       259

    accuracy                           0.96      1104
   macro avg       0.94      0.94      0.94      1104
weighted avg      

Traceback (most recent call last):
  File "/tmp/ipykernel_19/2255304482.py", line 609, in generate_waterfall_plot
    shap.waterfall_plot(explanation, show=False)
  File "/usr/local/lib/python3.8/site-packages/shap/plots/_waterfall.py", line 71, in waterfall
    base_values = float(shap_values.base_values)
TypeError: only size-1 arrays can be converted to Python scalars
Traceback (most recent call last):
  File "/tmp/ipykernel_19/2255304482.py", line 609, in generate_waterfall_plot
    shap.waterfall_plot(explanation, show=False)
  File "/usr/local/lib/python3.8/site-packages/shap/plots/_waterfall.py", line 71, in waterfall
    base_values = float(shap_values.base_values)
TypeError: only size-1 arrays can be converted to Python scalars
Traceback (most recent call last):
  File "/tmp/ipykernel_19/2255304482.py", line 609, in generate_waterfall_plot
    shap.waterfall_plot(explanation, show=False)
  File "/usr/local/lib/python3.8/site-packages/shap/plots/_waterfall.py", line 71, in water

ENSEMBLE - 10-Fold CV Results on Training Set:
  CV Accuracy: 0.9690
  CV F1 Macro: 0.9565
  CV F1 Weighted: 0.9689
ENSEMBLE - Final Test Results:
  Test Accuracy: 0.9547
  Test F1 Macro: 0.9368
  Test F1 Weighted: 0.9546

Classification Report for ENSEMBLE:
              precision    recall  f1-score   support

     Class 0       0.97      0.97      0.97       845
     Class 1       0.91      0.90      0.90       259

    accuracy                           0.95      1104
   macro avg       0.94      0.94      0.94      1104
weighted avg       0.95      0.95      0.95      1104

Saving models and results...
Model training and evaluation complete! Results saved to oxygen_model


In [ ]:

# Predict pathogenicity for a new genome using the ensemble model
prediction = predict_new_genome(
    genome_data="path/to/new_genome.csv",
    model_dir="./ntm_pathogenicity_model",
    model_type='ensemble'  # Use the ensemble of all three models
)